# 04 — Validate cohort membership

## Goal

For this portfolio project I use three source-defined culture cohorts: blood, respiratory, and urine. The mapping is deliberately simple and uses the exact ARMD `culture_description` values rather than trying to infer specimen types from text.

In [1]:
%sql
select
    cohort_id,
    count(*) as culture_organism_rows,
    count(distinct culture_key) as culture_orders,
    count(distinct patient_key) as patients,
    count(distinct organism) as source_organism_labels
from workspace.micro_dev.int_cohort_culture_organism
group by cohort_id
order by cohort_id;

,cohort_id,culture_organism_rows,culture_orders,patients,source_organism_labels
0,blood,13651,12728,10072,238
1,respiratory,22411,16738,7005,167
2,urine,97184,89301,53095,142


### Cohort size

The culture-organism counts are 13,651 for blood, 22,411 for respiratory, and 97,184 for urine. I keep culture-order and patient counts beside them so the difference between row grain and patient grain is visible.

In [2]:
%sql
select
    culture_organism_key,
    cohort_id,
    count(*) as row_count
from workspace.micro_dev.int_cohort_culture_organism
group by
    culture_organism_key,
    cohort_id
having count(*) > 1;

,culture_organism_key,cohort_id,row_count


### Grain check

No duplicate culture-organism/cohort combinations should be returned here.

In [3]:
%sql
select
    culture_description,
    count(*) as unmapped_culture_orders
from workspace.micro_dev.fct_culture
where culture_description not in (
    'BLOOD',
    'URINE',
    'RESPIRATORY'
)
group by culture_description
order by unmapped_culture_orders desc;

,culture_description,unmapped_culture_orders


### Coverage check

For the current release, every culture description used by the project maps to one of the three cohorts. I would rather fail this check on a future release than silently route a new source category into the wrong cohort.

In [4]:
%sql
select
    cohort_id,
    count(*) as culture_organism_rows,
    sum(
        case when was_positive = 1 then 1 else 0 end
    ) as positive_rows
from workspace.micro_dev.int_cohort_culture_organism
group by cohort_id
order by cohort_id;

,cohort_id,culture_organism_rows,positive_rows
0,blood,13651,13651
1,respiratory,22411,22411
2,urine,97184,97184


### Positive culture-organism rows

All rows in this cohort model are positive culture-organism records, which is consistent with the upstream organism fact.

### Takeaway

Cohort membership is explicit, reproducible, and does not depend on fuzzy specimen-name logic.